# Self-supervised finetuning on Xenium data

Adapt a pretrained TERRA model to **your own Xenium data** with **self-supervised LoRA fine-tuning** — no labels required — then embed and cluster with the adapted model. This continues TERRA's self-supervised pretraining objective on your data while training only small **LoRA adapters** on the frozen backbone: fast, data-efficient, and non-destructive.

**You will:**
1. Download a public single-cell-resolution Xenium sample (the same one as the quickstart).
2. Download a pretrained TERRA model from the Hugging Face Hub.
3. Tokenize the data for fine-tuning.
4. Fine-tune the encoder with self-supervised LoRA adapters.
5. Merge the adapters into a self-contained, embeddable model.
6. Embed with the adapted model and cluster.

:::{note}
**TERRA requires an NVIDIA GPU** for the tokenization, fine-tuning and embedding steps.
:::

New to TERRA? Start with the {doc}`zero-shot quickstart <zero_shot_quickstart>` first.

## 1. Setup

In [ ]:
# Colab / fresh environment only. Skip if you already installed TERRA per the installation guide.
%pip install terra-st

In [ ]:
import logging
from pathlib import Path

import numpy as np
import scanpy as sc

from terra import (download_pretrained, harmonize_adata, tokenize_adata,
                   harmonize_tokenize_embed_pipeline)
from terra.datasets import read_xenium_10x
from terra.training.finetune_self_supervised import (
    finetune_self_supervised, prepare_finetuned_model)

logging.basicConfig(level="INFO")  # see TERRA progress
sc.settings.set_figure_params(dpi=80, frameon=False)

## 2. Download the demo dataset

We use the 10x Genomics **Xenium Prime FFPE Human Skin** sample — the same public dataset as the {doc}`zero-shot quickstart <zero_shot_quickstart>`. Swap in your own raw-count Xenium `AnnData` (spatial coordinates in `obsm["spatial"]`) to fine-tune on your data.

In [ ]:
# Xenium Prime FFPE Human Skin (5,000-gene panel). The reader downloads two
# small standalone files (~44 MB) on first run and caches them.
adata = read_xenium_10x(
    "https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Human_Skin_FFPE/Xenium_Prime_Human_Skin_FFPE",
    "data/xenium_skin",
)
adata.obs["cell_id"] = adata.obs_names.astype(str)
adata.obs["sample"] = "skin"

# Crop a contiguous tissue window (so spatial neighborhoods stay intact) to keep
# the demo fast. Widen `half_window` (microns) to fine-tune on more cells.
xy = adata.obsm["spatial"]
center = np.median(xy, axis=0)
half_window = 400
adata = adata[(np.abs(xy - center) <= half_window).all(axis=1)].copy()
print(adata)

## 3. Download a pretrained TERRA model

`download_pretrained` fetches a self-contained model bundle (checkpoint, tokenizer, and the gene-reference files used for harmonization) and returns the local folder path.

In [ ]:
# By default the model is cached in the Hugging Face cache; pass
# local_dir="terra_model" to download it into a folder of your choice.
model_dir = download_pretrained("Lotfollahi-lab/TERRA-96M")

## 4. Tokenize for fine-tuning

Fine-tuning trains on **tokenized** data. We harmonize a copy of the sample (map genes to the model's Ensembl vocabulary + QC), tokenize it, and save it to disk. We keep the raw `adata` to embed later with the adapted model.

In [ ]:
# Reproduce the model's training-time gene mapping/filtering with the reference
# files shipped in the bundle.
gene_mapping_dict_file_path = str(Path(model_dir) / "ensembl_dictionary.pkl")
gene_occurrence_count_file_path = str(Path(model_dir) / "gene_count_dictionary.pkl")

adata_ft = harmonize_adata(
    adata.copy(),
    gene_mapping_dict_file_path=gene_mapping_dict_file_path,
    gene_occurrence_count_file_path=gene_occurrence_count_file_path,
    species="human",
    min_cells_per_gene=0,
    min_genes_per_cell=0,
)
dataset = tokenize_adata(adata_ft, model_folder_path=model_dir,
                         cache_directory_path="./terra_cache")

tokenised_dir = "data/xenium_skin_tokenised"
dataset.save_to_disk(tokenised_dir)
print(f"tokenised {len(dataset):,} cells -> {tokenised_dir}")

## 5. Fine-tune with LoRA (self-supervised)

Only the **LoRA adapters** on the attention/MLP projections are trained; the pretrained weights stay frozen. The config below mirrors the pretraining schedule, with a few epochs for this small demo crop — scale `num_epochs` and the data up for real adaptation.

In [ ]:
args = {
    "model": {
        "pretrained_checkpoint_path": model_dir,
        "finetune_checkpoint_path": "data/xenium_finetune",
    },
    "data": {
        "finetune_dataset": [tokenised_dir],
        "batch_size": 128,
        "num_workers": 8,
        "pin_memory": True,
        "drop_last": True,
        "sample_segments": False,
        "sample_gene_masks": True,
    },
    "finetune": {
        "num_epochs": 5,
        "lr": 1e-3,
        "start_lr": 1e-5,
        "final_lr": 1e-5,
        "warmup_epochs": 1,
        "weight_decay": 0.04,
        "final_weight_decay": 0.4,
        "ema_momentum": 0.9995,        # EMA for the target encoder
        "final_ema_momentum": 1.0,
        "loss_fn_type": "smooth_l1",
        "clip_grad": 2.0,
        "use_bfloat16": True,

        # LoRA (parameter-efficient fine-tuning)
        "use_peft": True,
        "peft_method": "lora",
        "peft_rank": 16,
        "peft_alpha": 256,
        "peft_dropout": 0.1,
        "peft_bias": "none",
        "peft_target_modules": ["qkv", "proj", "fc1", "fc2"],
        "save_every": 1,               # checkpoint every epoch
    },
}

# dataset=None -> the tokenised dataset listed in args["data"]["finetune_dataset"]
# is loaded automatically.
run_dir = finetune_self_supervised(args=args, dataset=None, run_name="xenium_finetune")

:::{note}
This is a tiny demo crop, so it runs quickly. Real adaptation uses more cells and
epochs and is a multi-hour GPU job.
:::

## 6. Merge the LoRA adapters

`prepare_finetuned_model` folds the adapters into the base weights and writes a self-contained model folder (`model_config.yaml`, `token_dictionary.pkl`, `model_checkpoint.pt`) that the embedding pipeline can consume directly.

In [ ]:
finetuned_model_dir = "data/xenium_lora_merged"
prepare_finetuned_model(
    finetuned_checkpoint_dir=run_dir,   # returned by finetune_self_supervised above
    pretrained_model_dir=model_dir,
    output_dir=finetuned_model_dir,
    use_peft=True,
)

## 7. Embed with the adapted model

`harmonize_tokenize_embed_pipeline` runs the full harmonize → tokenize → embed workflow in one call and stores the embeddings in `adata.obsm`. Point `model_folder_path` at the **adapted** model. (Point it at `model_dir` instead to compare against the pretrained model.)

In [ ]:
adata = harmonize_tokenize_embed_pipeline(
    adata=adata,
    sample_key=None,          # a single sample
    batch_key="sample",
    model_folder_path=finetuned_model_dir,
    cache_directory_path="./terra_cache",
    batch_size=128,           # lower this if you run out of GPU memory
)
# Embeddings now live in adata.obsm:
#   "cell_emb"         -> each cell's own expression
#   "neighborhood_emb" -> the cell together with its spatial neighborhood
print(list(adata.obsm))

## 8. Cluster & visualize

Cluster the **neighborhood** embedding to find spatial niches and the **cell** embedding to find cell types, then view each on its UMAP.

In [ ]:
for emb_key in ["cell_emb", "neighborhood_emb"]:
    sc.pp.neighbors(adata, use_rep=emb_key, key_added=emb_key)
    sc.tl.leiden(adata, neighbors_key=emb_key, key_added=emb_key + "_cluster",
                 resolution=0.6, flavor="igraph", n_iterations=2, directed=False)
    sc.tl.umap(adata, neighbors_key=emb_key)
    sc.pl.umap(adata, neighbors_key=emb_key, color=emb_key + "_cluster",
               title=emb_key)

## Next steps

- {doc}`supervised_finetuning` — fine-tune with a classification head for label transfer.
- {doc}`visium_transfer_learning_tutorial` — adapt the model across platforms (Xenium → Visium).
- {doc}`downstream_analysis` — gene-level embeddings, spatial gene-pair scoring, and perturbation.